In [7]:
# Sezione: import delle librerie necessarie
#
# Si usano:
# - networkx per la rappresentazione del grafo
# - numpy per i vettori di attivazione
# - deque per le visite in ampiezza
# - matplotlib per eventuali visualizzazioni

import networkx as nx
import numpy as np
from collections import deque
import matplotlib.pyplot as plt

# Helper

In [8]:
# Sezione: funzioni ausiliarie
#
# - conversione di un seed set in vettore binario di attivazione
# - calcolo della cardinalità finale e della traiettoria della diffusione

def make_vector(seed_set, n):
    x = np.zeros(n)
    x[list(seed_set)] = 1
    return x

# Diffusion Model

In [9]:
# Sezione: dinamica di diffusione

def connected_component_update(g, x, thetas):
    new_x = x.copy()
    n = len(x)

    active = set(np.where(x > 0)[0])

    for v in range(n):
        if x[v] == 1:
            continue

        reachable = 0
        visited = set()

        for u in g.neighbors(v):
            if u in active and u not in visited:
                comp = nx.node_connected_component(g.subgraph(active), u)
                reachable += len(comp)
                visited |= comp

        if reachable >= thetas[v] - 1:
            new_x[v] = 1

    return new_x


def connected_component_spread(g, x0, thetas, max_t=1000):
    x = x0.copy()
    spread_hist = [int(np.sum(x))]

    for _ in range(max_t):
        new_x = connected_component_update(g, x, thetas)

        if np.array_equal(new_x, x):
            break

        x = new_x
        spread_hist.append(int(np.sum(x)))

    return int(np.sum(x)), spread_hist, x



# Gamma

In [38]:
# Sezione: costruzione delle strutture per Gamma

# Sezione: Gamma(theta_i, S)
#
# Per ogni livello di soglia distinto theta_i, si considera il sottografo
# indotto dai vertici con soglia al più theta_i.
#
# La quantità Gamma(theta_i, S) è l'unione delle componenti connesse,
# in tale sottografo, che contengono almeno un vertice di S.

def build_gamma_data(g, thetas):
    n = g.number_of_nodes()

    levels = sorted(set(thetas))
    comp_id_by_level = []
    comp_size_by_level = []

    for level in levels:
        allowed = [v for v in g.nodes() if thetas[v] <= level]
        H = g.subgraph(allowed)

        comp_id = [-1] * n
        comp_size = {}

        cid = 0
        for comp in nx.connected_components(H):
            comp = list(comp)
            for v in comp:
                comp_id[v] = cid
            comp_size[cid] = len(comp)
            cid += 1

        comp_id_by_level.append(comp_id)
        comp_size_by_level.append(comp_size)

    return levels, comp_id_by_level, comp_size_by_level


    # : calcolo di |Gamma(theta_i, S)|

def gamma_size(g, thetas, S, level_idx, levels, comp_id_by_level, comp_size_by_level):
    level = levels[level_idx]

    touched_components = set()
    high_nodes = set()

    for v in S:
        if thetas[v] <= level:
            cid = comp_id_by_level[level_idx][v]
            if cid != -1:
                touched_components.add(cid)
        else:
            high_nodes.add(v)
            for u in g.neighbors(v):
                cid = comp_id_by_level[level_idx][u]
                if cid != -1:
                    touched_components.add(cid)

    total = len(high_nodes)
    total += sum(comp_size_by_level[level_idx][cid] for cid in touched_components)

    return total

# Funzione submodulare F

In [39]:
# Sezione: implementazione della funzione submodulare f

def f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level):
    l = len(levels)

    if l == 1:
        return max(0, min(len(S), levels[0] - 1))

    total = 0
    for i in range(l - 1):
        target = levels[i + 1] - 1
        total += min(
            gamma_size(g, thetas, S, i, levels, comp_id_by_level, comp_size_by_level),
            target
        )

    return total


def f_target(levels):
    l = len(levels)

    if l == 1:
        return max(0, levels[0] - 1)

    return sum(levels[i + 1] - 1 for i in range(l - 1))

# Greedy Algorithm

In [40]:
def greedy_seed_set(g, thetas, levels, comp_id_by_level, comp_size_by_level):
    S = set()
    target = f_target(levels)
    remaining = set(g.nodes())

    while f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level) < target:
        best_v = None
        best_gain = -1

        current_value = f_value(g, thetas, S, levels, comp_id_by_level, comp_size_by_level)

        for v in remaining:
            new_value = f_value(g, thetas, S | {v}, levels, comp_id_by_level, comp_size_by_level)
            gain = new_value - current_value

            if gain > best_gain:
                best_gain = gain
                best_v = v

        if best_v is None or best_gain <= 0:
            break

        S.add(best_v)
        remaining.remove(best_v)

    return S

# Connessione del Seedset

In [42]:
# Sezione: shortest path e connessione del seed set

def shortest_path_to_set(g, source, targets):
    if source in targets:
        return [source]

    visited = {source}
    parent = {source: None}
    queue = deque([source])

    while queue:
        u = queue.popleft()

        if u in targets:
            path = []
            while u is not None:
                path.append(u)
                u = parent[u]
            return path[::-1]

        for w in g.neighbors(u):
            if w not in visited:
                visited.add(w)
                parent[w] = u
                queue.append(w)

    return []


def connect_seed_set(g, D):
    D = list(D)

    if not D:
        return set()

    A = {D[0]}

    for v in D[1:]:
        path = shortest_path_to_set(g, v, A)
        A.update(path)

    return A

# Wrapper

In [43]:
def TD_orlogn(g, thetas, verbose=True):
    levels, comp_id_by_level, comp_size_by_level = build_gamma_data(g, thetas)

    D = greedy_seed_set(g, thetas, levels, comp_id_by_level, comp_size_by_level)
    A = connect_seed_set(g, D)

    spread, history, x = connected_component_spread(
        g,
        make_vector(A, g.number_of_nodes()),
        thetas
    )

    success = (spread == g.number_of_nodes())

    if verbose:
        print("Nodes:", g.number_of_nodes())
        print("Thresholds:", sorted(set(thetas)))
        print("D:", sorted(D))
        print("D size:", len(D))
        print("Seed set:", sorted(A))
        print("Seed set size:", len(A))
        print("Spread:", spread)
        print("Success:", success)

    return {
        "D": D,
        "A": A,
        "spread": spread,
        "history": history,
        "x": x,
        "success": success,
        "levels": levels
    }

# G&L

In [46]:
import math
import numpy as np
import networkx as nx

def assign_thresholds_paper_experiments(
    G: nx.Graph,
    c: int,
    seed: int | None = None,
    attr: str = "theta",
):
    """
    theta(v) uniformly random from {max(2,c), 2c, 3c, ..., ceil(n/c)*c}
    where n = number of nodes (paper uses n=200).
    Stores in G.nodes[v][attr] and returns (theta_dict, allowed_values).
    """
    if c <= 0:
        raise ValueError("c must be a positive integer.")
    rng = np.random.default_rng(seed)
    n = G.number_of_nodes()

    k_max = math.ceil(n / c)
    allowed_values = list(range(max(2, c), k_max * c + 1, c))

    theta = {v: int(rng.choice(allowed_values)) for v in G.nodes()}
    nx.set_node_attributes(G, theta, name=attr)
    return theta, allowed_values

In [56]:
N = 200
GRAPH_SEED = 42
THRESHOLD_SEED = 123
C = 1

g200 = preferential_attachment_variable_m(
    N=N,
    seed=GRAPH_SEED,
    init_nodes=5,
    init_mode="complete",
    M=(1, 2, 3, 4),
)

theta_dict_200, allowed_values_200 = assign_thresholds_paper_experiments(
    g200,
    c=C,
    seed=THRESHOLD_SEED,
    attr="theta",
)

thetas200 = np.array([theta_dict_200[i] for i in range(N)], dtype=int)

print("Allowed values:", allowed_values_200)
print("Distinct thresholds in instance:", sorted(set(thetas200)))
print("Min theta:", thetas200.min())
print("Max theta:", thetas200.max())

Allowed values: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200]
Distinct thresholds in instance: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64

In [57]:
res200 = TD_orlogn(g200, thetas200, verbose=True)

Nodes: 200
Thresholds: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(12), np.int64(13), np.int64(15), np.int64(16), np.int64(17), np.int64(20), np.int64(25), np.int64(27), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(42), np.int64(44), np.int64(45), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(57), np.int64(58), np.int64(60), np.int64(61), np.int64(62), np.int64(64), np.int64(65), np.int64(66), np.int64(68), np.int64(69), np.int64(71), np.int64(74), np.int64(75), np.int64(76), np.int64(77), np.int64(80), np.int64(82), np.int64(83), np.int64(84), np.int64(86), np.int64(87), np.int64(88), np.int64(89), np.int64(90), np.int64(91), np.int64(92), np.int64(94), np.int64(95), np.int64(98), np.int64(99), np.int64(100), np.int64(101), np.int64(102), np.int64(104), np.int64(105